# Sanskrit Character-LM — V1
**Model:** 6-layer decoder-only Transformer, 512-dim, flash attention, ~19M parameters  
**Data:** Full SLP1 corpus (171 MB), 90/10 train/val split  
**Target:** Converged character-level loss; internal activations that reflect Sanskrit phonological structure  
**Runtime:** Training spans multiple Colab sessions via Drive checkpointing — resume is automatic.

> Run cells top to bottom. If the session disconnects, re-run **Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4** to resume from the latest checkpoint automatically.

---
## Cell 0 · Config
All knobs in one place. Edit here; nothing else needs to change.

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
DATA_PATH  = '/content/drive/MyDrive/sanskrit/corpus.slp1.txt'
OUTPUT_DIR = '/content/drive/MyDrive/sanskrit/v1_run'

# ── DATA ───────────────────────────────────────────────────────────────────────
# None = read the entire file. Set an integer (e.g. 20_000_000) to cap during debugging.
MAX_CHARS  = None
TRAIN_FRAC = 0.90
# NOTE: this is a contiguous 90/10 split, which is fine for V1.
# Production runs should use work-level splits (index.csv maps each line to text_id).

# ── MODEL ──────────────────────────────────────────────────────────────────────
N_LAYER    = 6
N_HEAD     = 8
N_EMBD     = 512
BLOCK_SIZE = 512     # context window in tokens
DROPOUT    = 0.1

# ── TRAINING ───────────────────────────────────────────────────────────────────
BATCH_SIZE    = 64       # reduce to 32 if CUDA OOM
ACCUM_STEPS   = 1        # increase to simulate larger batch without more VRAM
MAX_ITERS     = 50_000   # designed for multi-session; checkpoint resumes automatically
LEARNING_RATE = 3e-4     # peak LR; decays to MIN_LR via cosine schedule
MIN_LR        = 3e-5     # cosine decay floor (~10% of peak)
WARMUP_ITERS  = 1_000    # linear warmup steps
EVAL_INTERVAL = 500      # eval train+val loss every N iters
EVAL_ITERS    = 100      # batches averaged per loss estimate
CKPT_INTERVAL = 1_000    # save rolling checkpoint every N iters

# ── MISC ───────────────────────────────────────────────────────────────────────
SEED          = 42
EXTRACT_LAYER = N_LAYER - 1  # layer to probe during activation extraction (0-indexed)

print('V1 config loaded.')
print(f'  Model      : {N_LAYER}L × {N_HEAD}H × {N_EMBD}D, ctx={BLOCK_SIZE}')
print(f'  Training   : {MAX_ITERS:,} iters, bs={BATCH_SIZE}×{ACCUM_STEPS}')
print(f'  LR         : {LEARNING_RATE} → {MIN_LR} (cosine)')
print(f'  Data       : {DATA_PATH}')
print(f'  Output     : {OUTPUT_DIR}')

---
## Cell 1 · Keep-Alive · Mount Drive · Setup · Device Check
The keep-alive cell pings the Colab UI every 55 seconds so the session doesn't
time out mid-training. Run it, then immediately run the rest of the cells — you
don't need to wait for it to finish.

In [ ]:
# ── Keep-alive: prevents Colab from disconnecting during long training runs ────
# Run this cell first, then proceed immediately to the next cell.
from IPython.display import display, Javascript
display(Javascript("""
(function keepAlive() {
  console.log('[keep-alive] ping');
  // Try both current and legacy Colab button selectors
  const selectors = [
    'colab-connect-button',
    'colab-toolbar-button#connect'
  ];
  for (const sel of selectors) {
    const el = document.querySelector(sel);
    if (el) { el.click(); break; }
  }
  setTimeout(keepAlive, 55000);
})();
"""))
print('Keep-alive started (pings every 55s). Proceed to the next cell.')

In [ ]:
# ── Mount Drive ────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Mixed-precision (fp16 autocast + GradScaler) ───────────────────────────────
try:
    from torch.amp import autocast as _ac, GradScaler as _GS
    def autocast(): return _ac(device_type='cuda', dtype=torch.float16)
    def make_scaler(): return _GS(device='cuda')
except Exception:
    from torch.cuda.amp import autocast as _ac, GradScaler as _GS
    def autocast(): return _ac()
    def make_scaler(): return _GS()

# ── Seed ──────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'Seed: {SEED}')

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.manual_seed(SEED)
    props = torch.cuda.get_device_properties(0)
    print(f'GPU  : {props.name}')
    print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
    # Flash attention requires PyTorch 2.0+
    flash_ok = hasattr(F, 'scaled_dot_product_attention')
    print(f'Flash attention : {"available" if flash_ok else "NOT available (PyTorch < 2.0)"}')
else:
    device  = torch.device('cpu')
    flash_ok = False
    print('WARNING: No GPU — CPU training will be extremely slow.')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output : {OUTPUT_DIR}')

---
## Cell 2 · Load Corpus · Vocab · Encode · Split
Reads the full SLP1 corpus from Drive, builds a character vocabulary from whatever
characters are present (no hardcoded alphabet), encodes to a compact int16 array,
and does a contiguous 90/10 train/val split.

The encoded array lives in CPU RAM (~350 MB for the full corpus). The high-RAM
runtime handles this easily; a standard runtime would too.

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
print(f'Reading {DATA_PATH} ...')
t0 = time.time()
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read(MAX_CHARS)   # MAX_CHARS=None reads everything
print(f'Loaded : {len(text):,} chars ({len(text)/1e6:.1f} MB) in {time.time()-t0:.1f}s')

# ── Vocabulary ────────────────────────────────────────────────────────────────
chars      = sorted(set(text))
vocab_size = len(chars)
stoi       = {c: i for i, c in enumerate(chars)}
itos       = {i: c for i, c in enumerate(chars)}

print(f'Vocab  : {vocab_size} characters')
print(f'Chars  : {repr("".join(chars))}')

# Save vocab to Drive so we can encode new text in future sessions
vocab_path = os.path.join(OUTPUT_DIR, 'vocab.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump({
        'stoi'      : stoi,
        'itos'      : {str(k): v for k, v in itos.items()},
        'vocab_size': vocab_size,
    }, f, ensure_ascii=False, indent=2)
print(f'Vocab saved → {vocab_path}')

# ── Encode ────────────────────────────────────────────────────────────────────
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda ids: ''.join(itos.get(i, '?') for i in ids)

print('Encoding corpus ...')
t0   = time.time()
# int16 supports up to 65535 IDs — more than enough for a char-level vocab
data = np.array(encode(text), dtype=np.int16)
del text   # free the raw string; the encoded array is all we need
print(f'Encoded: {len(data):,} tokens  ({data.nbytes/1e6:.0f} MB)  in {time.time()-t0:.1f}s')

# ── Train / val split ─────────────────────────────────────────────────────────
# Contiguous split: first 90% train, last 10% val.
# Production note: replace with work-level split using index.csv for no leakage.
n_train    = int(TRAIN_FRAC * len(data))
train_data = data[:n_train]
val_data   = data[n_train:]
print(f'Train  : {len(train_data):,} tokens  ({len(train_data)/1e6:.1f}M)')
print(f'Val    : {len(val_data):,} tokens  ({len(val_data)/1e6:.1f}M)')
print(f'Approx epoch length at bs={BATCH_SIZE}, ctx={BLOCK_SIZE}: '
      f'{len(train_data) / (BATCH_SIZE * BLOCK_SIZE):,.0f} iters')

# ── Batch sampler ─────────────────────────────────────────────────────────────
def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(d) - BLOCK_SIZE - 1, size=BATCH_SIZE)
    # Cast to int64 for embedding lookup; int16 storage is just to save RAM
    x  = torch.stack([torch.from_numpy(d[i  :i+BLOCK_SIZE  ].astype(np.int64)) for i in ix]).to(device)
    y  = torch.stack([torch.from_numpy(d[i+1:i+BLOCK_SIZE+1].astype(np.int64)) for i in ix]).to(device)
    return x, y

---
## Cell 3 · Model Definition
Decoder-only Transformer with three V1 upgrades over the feasibility version:
1. **Flash attention** (`F.scaled_dot_product_attention`) — memory-efficient, no explicit
   attention matrix allocation, falls back to manual attention on PyTorch < 2.0.
2. **Weight tying** — output head shares weights with the token embedding, reducing
   parameters and improving convergence (standard in GPT-2 and beyond).
3. **`return_hidden_states`** — built into `forward()` from the start so per-layer
   residual-stream tensors are always extractable for probing.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head   = n_head
        self.head_dim = n_embd // n_head
        self.dropout  = dropout
        self.c_attn   = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj   = nn.Linear(n_embd, n_embd,     bias=False)
        self.resid_drop = nn.Dropout(dropout)
        self.use_flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.use_flash:
            # Fallback: materialise the causal mask once
            self.register_buffer(
                'mask', torch.tril(torch.ones(block_size, block_size))
                              .unsqueeze(0).unsqueeze(0)
            )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        reshape  = lambda t: t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v  = reshape(q), reshape(k), reshape(v)

        if self.use_flash:
            # Flash attention: fused, memory-efficient, no O(T²) allocation
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask  = None,
                dropout_p  = self.dropout if self.training else 0.0,
                is_causal  = True,
            )
        else:
            # Manual fallback (PyTorch < 2.0)
            scale = self.head_dim ** -0.5
            att   = (q @ k.transpose(-2, -1)) * scale
            att   = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
            att   = torch.softmax(att.float(), dim=-1).to(q.dtype)
            out   = att @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class SanskritLM(nn.Module):
    """
    Decoder-only character-level Transformer for SLP1 Sanskrit.

    forward(idx, targets=None, return_hidden_states=False)
      return_hidden_states=True → also returns list[Tensor(B,T,D)] of
      residual-stream states after each block, for activation probing.
    """
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb    = nn.Embedding(vocab_size, n_embd)
        self.pos_emb    = nn.Embedding(block_size, n_embd)
        self.drop       = nn.Dropout(dropout)
        self.blocks     = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        # Weight tying: output projection shares weights with token embedding.
        # Reduces params and aligns input/output representations.
        self.head.weight = self.tok_emb.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None, return_hidden_states=False):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x   = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        hidden_states = []
        for block in self.blocks:
            x = block(x)
            if return_hidden_states:
                hidden_states.append(x.detach().clone())  # (B, T, n_embd)

        x      = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        if return_hidden_states:
            return logits, loss, hidden_states
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressive generation with optional top-k sampling."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt   = torch.multinomial(probs, num_samples=1)
            idx   = torch.cat([idx, nxt], dim=1)
        return idx


# ── Build model ───────────────────────────────────────────────────────────────
model = SanskritLM(
    vocab_size = vocab_size,
    n_layer    = N_LAYER,
    n_head     = N_HEAD,
    n_embd     = N_EMBD,
    block_size = BLOCK_SIZE,
    dropout    = DROPOUT,
).to(device)

# Parameter count (weight tying means tok_emb and head share one matrix)
n_params       = sum(p.numel() for p in model.parameters())
n_params_unique = sum(p.numel() for p in set(model.parameters()))
print(f'Parameters : {n_params_unique:,} unique  ({n_params_unique/1e6:.2f}M)')
print(f'             ({n_params:,} counted with tying)')

# Optional: torch.compile() for ~20-30% throughput gain on PyTorch 2.0+
if hasattr(torch, 'compile'):
    try:
        model = torch.compile(model)
        print('torch.compile() : enabled')
    except Exception as e:
        print(f'torch.compile() : skipped ({e})')
else:
    print('torch.compile() : not available (PyTorch < 2.0)')

---
## Cell 4 · Training Loop
- **Cosine LR decay**: rises linearly during warmup, then decays to `MIN_LR`.
- **Two checkpoints**: `checkpoint.pt` (rolling, every `CKPT_INTERVAL` iters) and
  `best_checkpoint.pt` (saved whenever val loss improves). Resume is automatic.
- **Best-model marker** `← best` appears in the log whenever a new val-loss minimum is set.
- If the session disconnects, re-run Cells 0–4 and training picks up where it left off.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95)
)
scaler = make_scaler()

# ── Cosine LR schedule with linear warmup ─────────────────────────────────────
def get_lr(it):
    if it < WARMUP_ITERS:
        return LEARNING_RATE * (it + 1) / WARMUP_ITERS
    if it >= MAX_ITERS:
        return MIN_LR
    progress = (it - WARMUP_ITERS) / (MAX_ITERS - WARMUP_ITERS)
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))

# ── Training state ────────────────────────────────────────────────────────────
start_iter    = 0
train_losses  = []   # [(iter, loss), ...]
val_losses    = []
tokens_seen   = 0
best_val_loss = float('inf')

ckpt_path      = os.path.join(OUTPUT_DIR, 'checkpoint.pt')
best_ckpt_path = os.path.join(OUTPUT_DIR, 'best_checkpoint.pt')

# ── Auto-resume ───────────────────────────────────────────────────────────────
if os.path.exists(ckpt_path):
    print(f'Checkpoint found — resuming ...')
    ckpt = torch.load(ckpt_path, map_location=device)
    # torch.compile wraps the model; load into the underlying module
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    raw_model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_iter    = ckpt['iter'] + 1
    train_losses  = ckpt.get('train_losses',  [])
    val_losses    = ckpt.get('val_losses',    [])
    tokens_seen   = ckpt.get('tokens_seen',   0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resumed at iter {start_iter:,}  '
          f'(best val loss so far: {best_val_loss:.4f})')
else:
    print('No checkpoint found — starting fresh.')

# ── Loss estimator ────────────────────────────────────────────────────────────
@torch.no_grad()
def estimate_loss():
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
    raw.eval()
    out = {}
    for split in ('train', 'val'):
        vals = []
        for _ in range(EVAL_ITERS):
            x, y = get_batch(split)
            with autocast():
                _, loss = model(x, y)
            vals.append(loss.item())
        out[split] = float(np.mean(vals))
    raw.train()
    return out

# ── Checkpoint helper ─────────────────────────────────────────────────────────
def save_ckpt(path, it):
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
    torch.save({
        'model'        : raw.state_dict(),
        'optimizer'    : optimizer.state_dict(),
        'iter'         : it,
        'tokens_seen'  : tokens_seen,
        'best_val_loss': best_val_loss,
        'config': dict(
            N_LAYER=N_LAYER, N_HEAD=N_HEAD, N_EMBD=N_EMBD,
            BLOCK_SIZE=BLOCK_SIZE, DROPOUT=DROPOUT, vocab_size=vocab_size,
        ),
        'vocab_stoi'   : stoi,
        'vocab_itos'   : itos,
        'train_losses' : train_losses,
        'val_losses'   : val_losses,
    }, path)

# ── Training ──────────────────────────────────────────────────────────────────
log_path = os.path.join(OUTPUT_DIR, 'train_log.txt')
t_start  = time.time()

print(f'\nTraining {MAX_ITERS:,} iters  (start={start_iter:,})'
      f'  bs={BATCH_SIZE}×{ACCUM_STEPS}  ctx={BLOCK_SIZE}')
print(f'Approx epochs per 10k iters: '
      f'{BATCH_SIZE * BLOCK_SIZE * 10_000 / len(train_data):.2f}')

try:
    for it in range(start_iter, MAX_ITERS):

        # ── Evaluate at iter 0 and every EVAL_INTERVAL iters ─────────────────
        if it % EVAL_INTERVAL == 0:
            losses  = estimate_loss()
            elapsed = time.time() - t_start
            tok_s   = tokens_seen / max(elapsed, 1e-9)
            lr_now  = get_lr(it)

            is_best = losses['val'] < best_val_loss
            if is_best:
                best_val_loss = losses['val']
                save_ckpt(best_ckpt_path, it)

            line = (f'iter {it:6,}  '
                    f'train {losses["train"]:.4f}  '
                    f'val {losses["val"]:.4f}'
                    + ('  ← best' if is_best else '       ')
                    + f'  lr {lr_now:.2e}  '
                    + f'tok/s {tok_s:>8,.0f}  '
                    + f'elapsed {elapsed/60:5.1f}m')
            print(line)
            with open(log_path, 'a') as lf:
                lf.write(line + '\n')
            train_losses.append((it, losses['train']))
            val_losses.append((it,   losses['val']))

        # ── Set LR for this iter ──────────────────────────────────────────────
        lr = get_lr(it)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # ── Forward / backward with gradient accumulation ─────────────────────
        optimizer.zero_grad(set_to_none=True)
        try:
            for _ in range(ACCUM_STEPS):
                x, y = get_batch('train')
                with autocast():
                    _, loss = model(x, y)
                scaler.scale(loss / ACCUM_STEPS).backward()
                tokens_seen += BATCH_SIZE * BLOCK_SIZE
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raise RuntimeError(
                f'CUDA OOM at iter {it}. '
                f'Lower BATCH_SIZE ({BATCH_SIZE} → 32) or BLOCK_SIZE ({BLOCK_SIZE} → 256) '
                f'in Cell 0, then re-run from Cell 0. The checkpoint is safe.'
            )

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # ── Rolling checkpoint ────────────────────────────────────────────────
        if (it > start_iter and it % CKPT_INTERVAL == 0) or it == MAX_ITERS - 1:
            save_ckpt(ckpt_path, it)
            print(f'  [checkpoint @ iter {it:,}]')

except KeyboardInterrupt:
    print('\nInterrupted — saving checkpoint.')
    save_ckpt(ckpt_path, it)

# ── Final numbers ─────────────────────────────────────────────────────────────
wall_time   = time.time() - t_start
final_train = train_losses[-1][1] if train_losses else float('nan')
final_val   = val_losses[-1][1]   if val_losses   else float('nan')
print(f'\nFinished. Wall time this session: {wall_time/60:.1f} min')
print(f'  Final  train loss : {final_train:.4f}')
print(f'  Final  val   loss : {final_val:.4f}')
print(f'  Best   val   loss : {best_val_loss:.4f}  → {best_ckpt_path}')

---
## Cell 5 · Loss Curve
Can be re-run at any time — loads history from the checkpoint if needed,
so you can plot the full curve even across multiple sessions.

In [ ]:
# Load loss history from checkpoint if not already in memory
# (safe to run standalone after a session restart)
_plot_train = train_losses
_plot_val   = val_losses

if not _plot_train and os.path.exists(ckpt_path):
    print('Loading loss history from checkpoint ...')
    _ckpt       = torch.load(ckpt_path, map_location='cpu')
    _plot_train = _ckpt.get('train_losses', [])
    _plot_val   = _ckpt.get('val_losses',   [])

if not _plot_train:
    print('No loss history found. Run training first.')
else:
    iters_t, losses_t = zip(*_plot_train)
    iters_v, losses_v = zip(*_plot_val)

    _best_iter = iters_v[int(np.argmin(losses_v))]
    _best_loss = min(losses_v)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(iters_t, losses_t, label='train', linewidth=1.5)
    ax.plot(iters_v, losses_v, label='val',   linewidth=1.5, linestyle='--')
    ax.axvline(_best_iter, color='green', linestyle=':', alpha=0.7,
               label=f'best val ({_best_loss:.4f} @ iter {_best_iter:,})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title(f'Sanskrit char-LM V1 — loss curves')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()

    plot_path = os.path.join(OUTPUT_DIR, 'loss_curve.png')
    fig.savefig(plot_path, dpi=150)
    plt.show()
    print(f'Plot saved → {plot_path}')
    print(f'Best val loss : {_best_loss:.4f} at iter {_best_iter:,}')
    print(f'Trained iters : {max(iters_t):,} of {MAX_ITERS:,} ({100*max(iters_t)/MAX_ITERS:.1f}%)')

---
## Cell 6 · Load Best Model + Generation
Loads `best_checkpoint.pt` (lowest val loss) rather than the final weights.
Run this cell independently after training — it rebuilds the model from the checkpoint.

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
# Prefer best_checkpoint.pt (lowest val loss); fall back to rolling checkpoint.
_load_path = best_ckpt_path if os.path.exists(best_ckpt_path) else ckpt_path
print(f'Loading model from: {_load_path}')

_ckpt  = torch.load(_load_path, map_location=device)
_cfg   = _ckpt['config']
_at_iter = _ckpt['iter']
_best_v  = _ckpt.get('best_val_loss', float('nan'))

# Rebuild a clean model (no torch.compile wrapper) from the saved config
best_model = SanskritLM(
    vocab_size = _cfg['vocab_size'],
    n_layer    = _cfg['N_LAYER'],
    n_head     = _cfg['N_HEAD'],
    n_embd     = _cfg['N_EMBD'],
    block_size = _cfg['BLOCK_SIZE'],
    dropout    = 0.0,   # disable dropout for inference
).to(device)
best_model.load_state_dict(_ckpt['model'])
best_model.eval()

print(f'  Iter when saved : {_at_iter:,}')
print(f'  Best val loss   : {_best_v:.4f}')

# Restore vocab from checkpoint in case this cell runs after a session restart
_stoi = _ckpt.get('vocab_stoi', stoi)
_itos = {int(k): v for k, v in _ckpt.get('vocab_itos', {str(i): c for i, c in itos.items()}).items()}
_enc  = lambda s: [_stoi[c] for c in s if c in _stoi]
_dec  = lambda ids: ''.join(_itos.get(i, '?') for i in ids)

# ── Generate ──────────────────────────────────────────────────────────────────
PROMPT      = 'agnim'   # SLP1 seed; edit to try different prompts
GEN_TOKENS  = 500
TEMPERATURE = 0.8       # lower = more conservative; higher = more varied
TOP_K       = 40        # None to disable top-k filtering

seed_ids = torch.tensor([_enc(PROMPT)], dtype=torch.long, device=device)
gen_ids  = best_model.generate(seed_ids, max_new_tokens=GEN_TOKENS,
                                temperature=TEMPERATURE, top_k=TOP_K)
gen_text = _dec(gen_ids[0].cpu().tolist())

print(f'\n── GENERATION (temp={TEMPERATURE}, top_k={TOP_K}) ─────────────────────────')
print(gen_text)
print('────────────────────────────────────────────────────────────────────────')
print('NOTE: SLP1 output — one ASCII char per Sanskrit phoneme.')
print('Plausible phonological patterns = model has learned SLP1 statistics.')

---
## Cell 7 · Activation Extraction
Runs a probe sentence through the best model and extracts the residual-stream
activation at every layer. Saves all layers to Drive.
This is the interface the probing experiments will call.

In [ ]:
# best_model and _dec/_enc must be in scope (run Cell 6 first)
best_model.eval()

# Use the first BLOCK_SIZE tokens from val as a fixed probe
if 'val_data' in dir():
    _probe_ids_np = val_data[:_cfg['BLOCK_SIZE']].astype(np.int64)
else:
    # Fallback if running standalone: load from vocab
    _probe_ids_np = np.array(_enc('agnim Ile purohitaM yajYasya devam ftvijam'), dtype=np.int64)

probe_ids  = torch.tensor(_probe_ids_np, dtype=torch.long, device=device).unsqueeze(0)
probe_text = _dec(_probe_ids_np.tolist())

with torch.no_grad():
    with autocast():
        logits, _, hidden_states = best_model(probe_ids, return_hidden_states=True)

assert len(hidden_states) == _cfg['N_LAYER']
h      = hidden_states[EXTRACT_LAYER]   # (1, T, n_embd)
h_fp32 = h.cpu().float()

print('── ACTIVATION EXTRACTION ────────────────────────────────────────────────')
print(f'Layers returned   : {len(hidden_states)}')
print(f'Extracted layer   : {EXTRACT_LAYER} (0-indexed)')
print(f'Tensor shape      : {tuple(h.shape)}  = (batch, seq_len, n_embd)')
print()
print('Slice h[0, :5, :8]  (first 5 positions × first 8 dims):')
print(h_fp32[0, :5, :8].numpy().round(4))
print()
print('Input characters  :', repr(probe_text[:10]), '...')

assert h_fp32.isfinite().all(), 'NaN/Inf in activations — model may have diverged.'
assert h_fp32.abs().max() > 0,  'All-zero activations — something is wrong.'

act_path = os.path.join(OUTPUT_DIR, 'sample_activations.pt')
torch.save({
    'hidden_states' : [hs.cpu() for hs in hidden_states],
    'probe_text'    : probe_text,
    'extract_layer' : EXTRACT_LAYER,
    'model_config'  : _cfg,
    'checkpoint'    : _load_path,
}, act_path)
print(f'\nAll {len(hidden_states)} layer activations saved → {act_path}')
print('ACTIVATION EXTRACTION: PASSED ✓')

---
## Cell 8 · Compute Report
Throughput, VRAM, and extrapolation numbers for planning larger runs.

In [ ]:
peak_mem_gb  = torch.cuda.max_memory_allocated(device) / 1e9 if torch.cuda.is_available() else 0
toks_per_sec = tokens_seen / max(wall_time, 1e-9)
gpu_name     = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
n_params_m   = sum(p.numel() for p in set(best_model.parameters())) / 1e6

print('─' * 68)
print('COMPUTE REPORT')
print('─' * 68)
print(f'  GPU                  : {gpu_name}')
print(f'  Model (unique params): {n_params_m:.2f}M  ({N_LAYER}L × {N_HEAD}H × {N_EMBD}D)')
print(f'  Flash attention      : {"yes" if flash_ok else "no"}')
print(f'  Context / batch      : {BLOCK_SIZE} tokens × {BATCH_SIZE} (×{ACCUM_STEPS} accum)')
print(f'  Iters this session   : {MAX_ITERS - start_iter:,}')
print(f'  Tokens this session  : {tokens_seen:,}')
print(f'  Wall time            : {wall_time:.0f}s  ({wall_time/3600:.2f}h)')
print(f'  Throughput           : {toks_per_sec:,.0f} tok/s')
print(f'  Peak GPU memory      : {peak_mem_gb:.2f} GB')
print('─' * 68)

_epoch_tokens = len(train_data)
_secs_per_epoch = _epoch_tokens / max(toks_per_sec, 1)
print(f'\nEXTRAPOLATION:')
print(f'  Train tokens / epoch : {_epoch_tokens/1e6:.1f}M')
print(f'  Time per epoch       : {_secs_per_epoch/60:.0f} min')
print(f'  Epochs in 12h session: {12*3600 / _secs_per_epoch:.1f}')
print(f'  Total iters for 5ep  : {5 * _epoch_tokens // (BATCH_SIZE * BLOCK_SIZE):,}')

# ── Summary ───────────────────────────────────────────────────────────────────
_initial_train = train_losses[0][1] if len(train_losses) > 1 else float('nan')
_decreased     = final_train < _initial_train

print('\n' + '═' * 68)
print('SUMMARY')
print('═' * 68)
print(f"""
Loss decreased    : {'YES' if _decreased else 'NO — investigate'}
  Initial train   : {_initial_train:.4f}
  Final   train   : {final_train:.4f}
  Final   val     : {final_val:.4f}
  Best    val     : {best_val_loss:.4f}  (saved to best_checkpoint.pt)

Generation        : SLP1 output with top-k={TOP_K}, temp={TEMPERATURE}.
                    Inspect for plausible Sanskrit phonological patterns.

Activations       : {N_LAYER} residual-stream tensors extracted, shape
                    (1, {_cfg['BLOCK_SIZE']}, {_cfg['N_EMBD']}) each. Saved to Drive.

Compute           : {toks_per_sec:,.0f} tok/s, {wall_time/60:.1f} min this session,
                    {peak_mem_gb:.2f} GB peak VRAM on {gpu_name}.

Multi-session     : Training is resumable. Re-run Cells 0-4 after a
                    disconnect to continue from iter {_ckpt.get('iter', '?')}.
""")